In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout, Input

In [2]:
train_ds = keras.utils.image_dataset_from_directory(
    directory = 'Brain Tumor\Training',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (128,128)
)

test_ds = keras.utils.image_dataset_from_directory(
    directory = 'Brain Tumor\Testing',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (128,128)
)

Found 5712 files belonging to 4 classes.
Found 1311 files belonging to 4 classes.


In [3]:
def scale_down_px(image, label):
  image = tf.cast(image, tf.float32)/255
  return image, label

In [4]:
train_ds = train_ds.map(scale_down_px)
test_ds = test_ds.map(scale_down_px)

In [5]:
model = Sequential()

model.add(Input(shape=(128,128,3)))

model.add(Conv2D(32, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(64, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(128, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Flatten())

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(4, activation='softmax'))

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,605,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,702,052 (6.49 MB)

 Trainable params: 1,701,604 (6.49 MB)

 Non-trainable params: 448 (1.75 KB)

In [7]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [8]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint = ModelCheckpoint('model1.keras', monitor='val_loss', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

history = model.fit(train_ds, validation_data=test_ds, epochs=30, callbacks=[early_stop, checkpoint])

Epoch 1/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 580ms/step - accuracy: 0.5277 - loss: 1.8828
Epoch 1: val_loss improved from inf to 1.45820, saving model to model1.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 112s 606ms/step - accuracy: 0.5280 - loss: 1.8794 - val_accuracy: 0.3242 - val_loss: 1.4582
Epoch 2/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 570ms/step - accuracy: 0.6772 - loss: 0.8461
Epoch 2: val_loss improved from 1.45820 to 0.83978, saving model to model1.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 106s 593ms/step - accuracy: 0.6773 - loss: 0.8459 - val_accuracy: 0.6568 - val_loss: 0.8398
Epoch 3/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 542ms/step - accuracy: 0.7202 - loss: 0.7026
Epoch 3: val_loss did not improve from 0.83978
179/179 ━━━━━━━━━━━━━━━━━━━━ 101s 564ms/step - accuracy: 0.7203 - loss: 0.7025 - val_accuracy: 0.5652 - val_loss: 0.9842
Epoch 4/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 538ms/step - accuracy: 0.7669 - loss: 0.5684
Epoch 4: val_loss improved from 0.83978 to 0.45873, saving model to model1.keras

In [9]:
from keras.models import load_model
model = load_model("model1.keras")

In [10]:
loss, accuracy = model.evaluate(test_ds)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step - accuracy: 0.8914 - loss: 0.2408
Test Loss: 0.2466
Test Accuracy: 0.8894


In [11]:
model = Sequential()

model.add(Input(shape=(128,128,3)))

model.add(Conv2D(32, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(64, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(128, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(256, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Flatten())

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(4, activation='softmax'))

In [12]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │       589,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 982,436 (3.75 MB)

 Trainable params: 981,476 (3.74 MB)

 Non-trainable params: 960 (3.75 KB)

In [13]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [14]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint = ModelCheckpoint('model2.keras', monitor='val_loss', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

history = model.fit(train_ds, validation_data=test_ds, epochs=30, callbacks=[early_stop, checkpoint])

Epoch 1/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 587ms/step - accuracy: 0.5417 - loss: 1.3941
Epoch 1: val_loss improved from inf to 1.93267, saving model to model2.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 113s 612ms/step - accuracy: 0.5420 - loss: 1.3924 - val_accuracy: 0.2571 - val_loss: 1.9327
Epoch 2/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 597ms/step - accuracy: 0.7013 - loss: 0.7952
Epoch 2: val_loss improved from 1.93267 to 0.98534, saving model to model2.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 112s 626ms/step - accuracy: 0.7014 - loss: 0.7949 - val_accuracy: 0.6491 - val_loss: 0.9853
Epoch 3/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 611ms/step - accuracy: 0.7953 - loss: 0.6046
Epoch 3: val_loss improved from 0.98534 to 0.91486, saving model to model2.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 114s 636ms/step - accuracy: 0.7953 - loss: 0.6046 - val_accuracy: 0.6484 - val_loss: 0.9149
Epoch 4/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 585ms/step - accuracy: 0.7968 - loss: 0.5402
Epoch 4: val_loss improved from 0.91486 to 0.4

In [15]:
from keras.models import load_model
model = load_model("model2.keras")

In [16]:
loss, accuracy = model.evaluate(test_ds)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 109ms/step - accuracy: 0.9557 - loss: 0.1556
Test Loss: 0.1304
Test Accuracy: 0.9626


In [19]:
model = Sequential()

model.add(Input(shape=(128,128,3)))

model.add(Conv2D(16, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(32, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(64, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(128, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Flatten())

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(4, activation='softmax'))

In [20]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)               │ (None, 126, 126, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 126, 126, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 63, 63, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 61, 61, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 61, 61, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 28, 28, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 395,588 (1.51 MB)

 Trainable params: 395,108 (1.51 MB)

 Non-trainable params: 480 (1.88 KB)

In [21]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [22]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint = ModelCheckpoint('model3.keras', monitor='val_loss', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

history = model.fit(train_ds, validation_data=test_ds, epochs=30, callbacks=[early_stop, checkpoint])

Epoch 1/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.6124 - loss: 1.0636
Epoch 1: val_loss improved from inf to 1.96781, saving model to model3.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 51s 256ms/step - accuracy: 0.6129 - loss: 1.0620 - val_accuracy: 0.3242 - val_loss: 1.9678
Epoch 2/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.8056 - loss: 0.4933
Epoch 2: val_loss improved from 1.96781 to 1.56163, saving model to model3.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 46s 256ms/step - accuracy: 0.8057 - loss: 0.4932 - val_accuracy: 0.4813 - val_loss: 1.5616
Epoch 3/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.8387 - loss: 0.4143
Epoch 3: val_loss improved from 1.56163 to 0.49242, saving model to model3.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 46s 254ms/step - accuracy: 0.8388 - loss: 0.4141 - val_accuracy: 0.8085 - val_loss: 0.4924
Epoch 4/30
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.8719 - loss: 0.3376
Epoch 4: val_loss did not improve from 0.49242
17

In [23]:
from keras.models import load_model
model = load_model("model3.keras")

In [24]:
loss, accuracy = model.evaluate(test_ds)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

41/41 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.9631 - loss: 0.1496
Test Loss: 0.1499
Test Accuracy: 0.9603
